In [1]:
!pip install pypdf langchain-text-splitters chromadb sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import warnings
from pathlib import Path
from pypdf import PdfReader

# تعطيل رسائل التنبيه وأشرطة التقدم المسببة لتعليق الواجهة
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# مسار مجلد ملفات الـ PDF
DATA_DIR = Path(r"C:\Users\Ahmed\Desktop\NRAG\data")

def extract_text_from_pdfs(folder_path: Path) -> list[dict]:
    extracted_data = []
    
    # فحص الملفات والتأكد من عدم تكرار قراءة الـ PDF على نظام ويندوز
    pdf_files = sorted([
        f for f in folder_path.iterdir() 
        if f.is_file() and f.suffix.lower() == ".pdf"
    ])
    
    if not pdf_files:
        print(f"No PDF files found in: {folder_path}")
        return extracted_data

    print(f"Found {len(pdf_files)} PDF file(s). Extracting text...\n")

    for pdf_path in pdf_files:
        try:
            reader = PdfReader(pdf_path)
            total_pages = len(reader.pages)
            
            for page_num, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""
                cleaned_text = page_text.strip()
                
                if cleaned_text:
                    extracted_data.append({
                        "file_name": pdf_path.name,
                        "file_path": str(pdf_path),
                        "page_number": page_num,
                        "total_pages": total_pages,
                        "text": cleaned_text
                    })
            print(f"✓ Processed: {pdf_path.name} ({total_pages} pages)")
        except Exception as e:
            print(f"✗ Failed to read {pdf_path.name}: {e}")
            
    return extracted_data

# تشغيل الاستخراج
records = extract_text_from_pdfs(DATA_DIR)
print(f"\nTotal raw pages extracted: {len(records)}")

Found 7 PDF file(s). Extracting text...

✓ Processed: 01_Resume_Writing_Best_Practices.pdf (3 pages)
✓ Processed: 02_How_to_Analyze_a_Job_Description.pdf (3 pages)
✓ Processed: 03_Behavioral_Interview_Framework.pdf (3 pages)
✓ Processed: 04_Technical_Interview_and_System_Design.pdf (3 pages)
✓ Processed: 05_Salary_Negotiation_Playbook.pdf (2 pages)
✓ Processed: 06_Career_Growth_and_Personal_Branding.pdf (2 pages)
✓ Processed: 07_Data_Roles_Roadmaps_and_Tools.pdf (3 pages)

Total raw pages extracted: 19


In [4]:
records[0]

{'file_name': '01_Resume_Writing_Best_Practices.pdf',
 'file_path': 'C:\\Users\\Ahmed\\Desktop\\NRAG\\data\\01_Resume_Writing_Best_Practices.pdf',
 'page_number': 1,
 'total_pages': 3,
 'text': 'Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actually 

In [5]:
import re

def clean_document_text(text: str) -> str:
    # 1. إزالة التذييل والفقرات القانونية المكررة في نهايات الصفحات
    disclaimer_pattern = r"This guide reflects widely-observed hiring practices.*?industry, and location\."
    text = re.sub(disclaimer_pattern, "", text, flags=re.DOTALL | re.IGNORECASE)

    # 2. حذف رموز التحكم المخفية المشوهة للترميز (\x7f)
    text = text.replace('\x7f', '')

    # 3. دمج الكلمات المفصولة بشرطة نهاية السطر (e.g., "re-\nsponsible" -> "responsible")
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)

    # 4. توحيد علامات الترقيم والمربعات (■, •, ▪, ►, *) إلى نقطة موحدة في بداية السطر
    text = re.sub(r'(?:[\r\n]+\s*)+[■•▪►\*\-]\s*', r'\n- ', text)

    # 5. تنظيف أي علامات تنقيط يتيمة ناتجة عن التنسيق
    text = re.sub(r'\n[-*]\s*\n', '\n- ', text)

    # 6. ضبط المسافات وفواصل الأسطر الزائدة
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

def preprocess_records(raw_records: list[dict]) -> list[dict]:
    cleaned = []
    for record in raw_records:
        cleaned_content = clean_document_text(record["text"])
        if len(cleaned_content) > 30:
            cleaned.append({
                "file_name": record["file_name"],
                "file_path": record["file_path"],
                "page_number": record["page_number"],
                "total_pages": record["total_pages"],
                "text": cleaned_content
            })
    print(f"✓ Cleaned {len(cleaned)} pages successfully.")
    return cleaned

# تشغيل التنظيف
cleaned_records = preprocess_records(records)

✓ Cleaned 18 pages successfully.


In [7]:
cleaned_records[:3]

[{'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'file_path': 'C:\\Users\\Ahmed\\Desktop\\NRAG\\data\\01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'total_pages': 3,
  'text': 'Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actu

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# إعداد التقطيع مع تداخل كافٍ لضمان عدم فصل العناوين عن المعادلات والأمثلة
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = []
counter = 1

for record in cleaned_records:
    splits = text_splitter.split_text(record["text"])
    for s in splits:
        if len(s.strip()) > 40:
            chunks.append({
                "chunk_id": f"{record['file_name']}_p{record['page_number']}_c{counter}",
                "file_name": str(record["file_name"]),
                "page_number": int(record["page_number"]),
                "text": s.strip()
            })
            counter += 1

print(f"✓ Generated {len(chunks)} cohesive chunks with context overlap.")

✓ Generated 78 cohesive chunks with context overlap.


In [9]:
chunks[:3]

[{'chunk_id': '01_Resume_Writing_Best_Practices.pdf_p1_c1',
  'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'text': "Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter's 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a"},
 {'chunk_id': '01_Resume_Writing_Best_Practices.pdf_p1_c2',
  'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'text': 'A resume stu

In [10]:
import chromadb
from chromadb.utils import embedding_functions

# مسار مجلد تخزين قاعدة البيانات محلياً
CHROMA_PATH = r"C:\Users\Ahmed\Desktop\NRAG\chroma_db"
os.makedirs(CHROMA_PATH, exist_ok=True)

# تهيئة العميل الدائم
client = chromadb.PersistentClient(path=CHROMA_PATH)

# دالة التضمين باستخدام نموذج MiniLM الخفيف والسريع
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# حذف المجموعة السابقة إذا كانت موجودة لضمان عدم تلوث قاعدة البيانات بمقاطع قديمة
try:
    client.delete_collection(name="career_knowledge_base")
    print("Deleted old collection to prevent stale data.")
except Exception:
    pass

# إنشاء مجموعة جديدة ونظيفة بضبط مقياس Cosine Similarity
collection = client.create_collection(
    name="career_knowledge_base",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# إدخال المقاطع على دفعات (Batches)
batch_size = 64
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{"file_name": c["file_name"], "page_number": c["page_number"]} for c in batch]
    )

print(f"✓ Clean collection indexed successfully! Total chunks in DB: {collection.count()}")

Deleted old collection to prevent stale data.
✓ Clean collection indexed successfully! Total chunks in DB: 78


In [11]:
def retrieve_relevant_chunks(query: str, n_results: int = 4) -> list[dict]:
    """
    استرجاع أفضل المقاطع الدلالية ذات الصلة بسؤال المستخدم مع الميتا-داتا
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    
    retrieved_items = []
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]
    
    for doc, meta, dist in zip(docs, metas, distances):
        similarity = round((1 - dist) * 100, 2)
        retrieved_items.append({
            "text": doc,
            "file_name": meta["file_name"],
            "page_number": meta["page_number"],
            "similarity": similarity
        })
        
    return retrieved_items

def search_and_display(query: str, n_results: int = 4):
    """دالة منسقة لعرض نتائج البحث وقراءتها بسهولة"""
    print(f"\nSearching for: '{query}'")
    print("=" * 70)
    
    hits = retrieve_relevant_chunks(query, n_results=n_results)
    for idx, hit in enumerate(hits, start=1):
        print(f"Result #{idx} | Similarity: {hit['similarity']}% | Source: {hit['file_name']} (Page {hit['page_number']})")
        print("-" * 70)
        print(hit["text"])
        print("=" * 70)

# اختبار البحث بالسؤال
test_query = "How should I structure my resume bullet points to show measurable impact?"
search_and_display(test_query, n_results=4)


Searching for: 'How should I structure my resume bullet points to show measurable impact?'
Result #1 | Similarity: 58.56% | Source: 01_Resume_Writing_Best_Practices.pdf (Page 3)
----------------------------------------------------------------------
Mirror the job description's language truthfully in your summary and skills section (see the companion
"How to Analyze a Job Description" guide for the extraction process).

Reorder your bullets so the most relevant achievements for *this* role appear first within each job
entry.

Run your tailored resume through a free ATS/keyword checker against the specific posting before
submitting, if one is available to you.
Result #2 | Similarity: 53.86% | Source: 01_Resume_Writing_Best_Practices.pdf (Page 2)
----------------------------------------------------------------------
Projects / Portfolio (optional but increasingly expected in tech) -- a short section linking to GitHub
repos, a portfolio site, or published work.
3. Writing Bullets That Act

In [11]:
query = "How should I structure my resume bullet points to show measurable impact?"
print(f"\nSearching for: '{query}'\n" + "=" * 65)

results = collection.query(
    query_texts=[query],
    n_results=4,
    include=["documents", "metadatas", "distances"]
)

for idx, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1):
    similarity = round((1 - dist) * 100, 2)
    print(f"Result #{idx} | Similarity: {similarity}% | Source: {meta['file_name']} (Page {meta['page_number']})")
    print("-" * 65)
    print(doc[:350] + ("..." if len(doc) > 350 else ""))
    print("=" * 65)


Searching for: 'How should I structure my resume bullet points to show measurable impact?'
Result #1 | Similarity: 58.56% | Source: 01_Resume_Writing_Best_Practices.pdf (Page 3)
-----------------------------------------------------------------
Mirror the job description's language truthfully in your summary and skills section (see the companion
"How to Analyze a Job Description" guide for the extraction process).

Reorder your bullets so the most relevant achievements for *this* role appear first within each job
entry.

Run your tailored resume through a free ATS/keyword checker against ...
Result #2 | Similarity: 53.86% | Source: 01_Resume_Writing_Best_Practices.pdf (Page 2)
-----------------------------------------------------------------
Projects / Portfolio (optional but increasingly expected in tech) -- a short section linking to GitHub
repos, a portfolio site, or published work.
3. Writing Bullets That Actually Land
The single highest-leverage change most resumes need: replace d

In [12]:
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable


In [17]:
import ollama

def build_context(retrieved_chunks: list[dict]) -> str:
    """
    تجميع وتنسيق السياق مع بيانات المصدر والصفحة
    """
    context_blocks = []
    for idx, chunk in enumerate(retrieved_chunks, start=1):
        block = (
            f"--- Document [{idx}] ---\n"
            f"Source File: {chunk['file_name']} (Page {chunk['page_number']})\n"
            f"Content:\n{chunk['text'].strip()}"
        )
        context_blocks.append(block)
    
    return "\n\n".join(context_blocks)


def generate_rag_response(query: str, n_results: int = 5) -> str:
    # 1. زيادة عدد النتائج إلى 5 لجلب كل تفاصيل القسم
    retrieved_chunks = retrieve_relevant_chunks(query, n_results=n_results)
    
    if not retrieved_chunks:
        return "لم يتم العثور على معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال."

    formatted_context = build_context(retrieved_chunks)

    # 2. التعديل هنا: توجيه النموذج للاعتراف بالصيغ والقواعد كهيكل أساسي
    system_instruction = (
        "You are an expert career consultant. Answer the user's question directly and comprehensively using ONLY the provided context.\n"
        "Strict Rules:\n"
        "1. NEVER invent, extrapolate, or fabricate any examples. If an example is provided in the text, quote or adapt ONLY that exact example.\n"
        "2. Present the X-Y-Z formula and its accompanying rules (verbs, tools in context, metrics) completely as stated.\n"
        "3. Every paragraph or piece of advice MUST end with an explicit source citation in the format: (01_Resume_Writing_Best_Practices.pdf, Page X).\n"
        "4. Do NOT refer to documents as 'Document [1]'; always use their real file names."
    )

    user_message = f"""Context Documents:
{formatted_context}

Question: {query}

Provide a structured, helpful answer based strictly on the context above:"""

    print("Generating refined answer using local llama3.2:3b...")
    
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message}
        ],
        options={
            "temperature": 0.1  # تثبيت الإجابة وتقليل التشتت
        }
    )

    return response["message"]["content"]

In [18]:
# ====================================================
# تجربة عملية كاملة (End-to-End Test)
# ====================================================
test_query = "How should I structure my resume bullet points to show measurable impact?"

answer = generate_rag_response(test_query, n_results=4)

print("\n" + "=" * 70)
print("🤖 Final Local RAG Response:")
print("=" * 70)
print(answer)

Generating refined answer using local llama3.2:3b...

🤖 Final Local RAG Response:
To structure your resume bullet points and show measurable impact, follow the X-Y-Z formula: Accomplished [X], measured by [Y], by doing [Z]. This formula is recommended in Document [2] (Page 2) as the single highest-leverage change most resumes need.

Here's a structured approach:

1. Identify the most relevant achievements for the role you're applying to.
2. Reorder your bullet points within each job entry so that the most JD-relevant achievements appear first.
3. Use the X-Y-Z formula to rephrase your bullet points, focusing on measurable outcomes and concrete actions.

Example: Instead of "Responsible for managing a team," use "Reduced team turnover by 25% (Y) by implementing a comprehensive onboarding program (Z), resulting in improved employee satisfaction (X)."

By following this approach, you'll be able to showcase your achievements in a clear and impactful way, making it easier for the hiring man